# Phase 05D_A3 - BiomedCLIP Dual-View Naturalized Global-23 Ensemble

This notebook runs the strongest clean BiomedCLIP baseline: raw MRI montage plus MONAI-overlay montage, naturalized answer-candidate text, and averaged dual-view candidate logits.

It includes `diagnostic`, `speed_test`, `screening`, and `full` modes. The prediction protocol is strict:
every question is scored against the same fixed 23-answer BTUMQA vocabulary, without
question-family filtering or row-level gold-answer fallback. Production modes use compact
outputs, single-prompt scoring, batched BiomedCLIP inference, and local `/content` montage staging.


## Dual Environment Compatibility Setup & Install Required Dependencies


In [ ]:
# ── DUAL ENVIRONMENT COMPATIBILITY & DEPENDENCY SETUP ────────────────────────
import os
import sys
from pathlib import Path

def resolve_project_environment(mount_point: str = "/content/drive") -> tuple[Path, Path]:
    try:
        import google.colab
        from google.colab import drive
        drive.mount(mount_point)
        project_root = Path(mount_point) / "MyDrive" / "AUGR-VQA"
        temp_dir = Path("/content")
        print("Running in Google Colab environment.")
    except ImportError:
        # Running locally (parent of notebooks directory)
        project_root = Path(os.getcwd()).parent.resolve()
        temp_dir = project_root / "temp"
        temp_dir.mkdir(parents=True, exist_ok=True)
        print("Running in Local environment.")
    return project_root, temp_dir

PROJECT_ROOT, TEMP_DIR = resolve_project_environment()
# ─────────────────────────────────────────────────────────────────────────────

import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "open_clip_torch",
    "timm",
    "ftfy",
    "regex",
    "tqdm",
    "scikit-learn==1.6.1",
    "pandas==2.2.2",
    "pillow==10.4.0",
])
print("BiomedCLIP dependencies installed with pandas 2.2.2, scikit-learn 1.6.1, and Pillow 10.4.0 pins.")


BiomedCLIP dependencies installed with pandas 2.2.2, scikit-learn 1.6.1, and Pillow 10.4.0 pins.


## Mount Drive And Locate Dataset


In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil
import time
import zipfile


def ensure_drive_connection(project_dir: Path, mount_point: str = "/content/drive"):
    def probe_path(target: Path):
        probe_target = target if target.exists() else target.parent
        return os.listdir(str(probe_target))

    try:
        probe_path(project_dir)
    except OSError as exc:
        if getattr(exc, "errno", None) != 107:
            raise
        print("Detected a stale Google Drive mount. Remounting now...")
        try:
            drive.flush_and_unmount()
            time.sleep(2)
        except Exception:
            pass
#         drive.mount(mount_point, force_remount=True)
        time.sleep(2)
        probe_path(project_dir)

    if not project_dir.exists():
        raise FileNotFoundError(f"Project Drive directory not found: {project_dir}")


# drive.mount("/content/drive")

PROJECT_DRIVE_DIR = PROJECT_ROOT
ensure_drive_connection(PROJECT_DRIVE_DIR)

DATASET_CSV_CANDIDATES = [
    PROJECT_DRIVE_DIR / "phase_3/p3a_brats_vqa_dataset" / "dataset_btumqa_225k",
    PROJECT_DRIVE_DIR / "Dataset" / "dataset_btumqa_225k",
    PROJECT_DRIVE_DIR / "dataset_btumqa_225k",
]

IMAGE_BASE_CANDIDATES = [
    PROJECT_DRIVE_DIR / "phase_1" / "p1a_segmentation_monai_brats" / "dataset_cache" / "rsnabrats20212d-001",
    PROJECT_DRIVE_DIR / "phase_1" / "p1a_segmentation_monai_brats" / "dataset_cache" / "rsna-brats-2021-2d",
    PROJECT_DRIVE_DIR / "Dataset" / "rsna-brats-2021-2d",
    PROJECT_DRIVE_DIR / "rsna-brats-2021-2d",
    PROJECT_DRIVE_DIR / "phase_0_data" / "rsna-brats-2021-2d",
]

DRIVE_DATASET_ZIP_CANDIDATES = [
    PROJECT_DRIVE_DIR / "phase_1/p1a_segmentation_monai_brats" / "dataset_cache" / "rsnabrats20212d.zip",
    PROJECT_DRIVE_DIR / "phase_2/p2b_mc_dropout_uncertainty" / "dataset_cache" / "rsnabrats20212d.zip",
    PROJECT_DRIVE_DIR / "phase_2/p2b_mc_dropout_uncertainty" / "dataset_cache" / "rsnabrats20212d.zip",
]

LOCAL_DATASET_DIR = TEMP_DIR / "phase05d_global23_rsnabrats20212d")
LOCAL_DATASET_ZIP = LOCAL_DATASET_DIR / "rsnabrats20212d.zip"
LOCAL_MONTAGE_DIR = TEMP_DIR / "phase05d_global23_montage_cache")
LOCAL_MASK_CACHE_DIR = TEMP_DIR / "phase05d_global23_monai_mask_cache")
LOCAL_MODEL_CACHE_DIR = TEMP_DIR / "phase05d_global23_model_cache")
LOCAL_MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(LOCAL_MODEL_CACHE_DIR / "huggingface")
os.environ["TRANSFORMERS_CACHE"] = str(LOCAL_MODEL_CACHE_DIR / "transformers")
os.environ["XDG_CACHE_HOME"] = str(LOCAL_MODEL_CACHE_DIR / "xdg")


def first_existing_dir(candidates, name):
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Could not find {name}. Checked:\n" + "\n".join(str(path) for path in candidates)
    )


def looks_like_image_base(path: Path):
    if not path.exists():
        return False
    modality_dirs = [path / modality for modality in ["flair", "t1", "t1ce", "t2"]]
    return all(modality_dir.exists() for modality_dir in modality_dirs) and any(
        (path / "flair").glob("flair_BraTS2021_*.png")
    )


def find_extracted_image_base(root: Path):
    if looks_like_image_base(root):
        return root
    if not root.exists():
        return None
    for candidate in root.rglob("*"):
        if candidate.is_dir() and looks_like_image_base(candidate):
            return candidate
    return None


def prepare_local_image_base():
    for candidate in IMAGE_BASE_CANDIDATES:
        if looks_like_image_base(candidate):
            print("Using existing image directory:", candidate)
            return candidate

    existing_local = find_extracted_image_base(LOCAL_DATASET_DIR)
    if existing_local is not None:
        print("Using existing local extracted image directory:", existing_local)
        return existing_local

    drive_zip = next((path for path in DRIVE_DATASET_ZIP_CANDIDATES if path.exists()), None)
    if drive_zip is None:
        raise FileNotFoundError(
            "Could not find RSNA-BraTS 2D image directory or cached dataset zip. "
            "Checked image dirs:\n"
            + "\n".join(str(path) for path in IMAGE_BASE_CANDIDATES)
            + "\n\nChecked zip caches:\n"
            + "\n".join(str(path) for path in DRIVE_DATASET_ZIP_CANDIDATES)
        )

    LOCAL_DATASET_DIR.mkdir(parents=True, exist_ok=True)
    if not LOCAL_DATASET_ZIP.exists() or LOCAL_DATASET_ZIP.stat().st_size != drive_zip.stat().st_size:
        print("Restoring RSNA-BraTS 2D zip from Drive cache:", drive_zip)
        shutil.copy2(drive_zip, LOCAL_DATASET_ZIP)
    else:
        print("Local RSNA-BraTS 2D zip already exists:", LOCAL_DATASET_ZIP)

    print("Extracting RSNA-BraTS 2D dataset locally. This may take several minutes...")
    with zipfile.ZipFile(LOCAL_DATASET_ZIP, "r") as zip_ref:
        zip_ref.extractall(LOCAL_DATASET_DIR)

    extracted_base = find_extracted_image_base(LOCAL_DATASET_DIR)
    if extracted_base is None:
        raise FileNotFoundError("Dataset zip was extracted, but no flair/t1/t1ce/t2 PNG folder was found.")
    print("Using extracted image directory:", extracted_base)
    return extracted_base


DATASET_DIR = first_existing_dir(DATASET_CSV_CANDIDATES, "BTUMQA-225K dataset directory")
IMAGE_BASE_DIR = prepare_local_image_base()

TEST_CSV_PATH = DATASET_DIR / "btumqa_225k_test.csv"
TRAIN_CSV_PATH = DATASET_DIR / "btumqa_225k_train.csv"
VAL_CSV_PATH = DATASET_DIR / "btumqa_225k_val.csv"
QA_PAIRS_CSV_PATH = DATASET_DIR / "btumqa_225k_qa_pairs.csv"

PHASE5D_BASE_DIR = PROJECT_DRIVE_DIR / "phase_5/p5d_modern_baseline_comparison_models"
PHASE5E_BASE_DIR = PROJECT_DRIVE_DIR / "phase_5/p5e_modern_vlm_baseline_comparison"

PHASE1_SEGMENTATION_DIR = PROJECT_DRIVE_DIR / "phase_1/p1a_segmentation_monai_brats"
PHASE1_MASK_DIR = PHASE1_SEGMENTATION_DIR / "generated_masks_rsna_2d"
PHASE1_MASK_MANIFEST_PATH = PHASE1_SEGMENTATION_DIR / "logs" / "global_mask_manifest.csv"

print("Project directory:", PROJECT_DRIVE_DIR)
print("Dataset directory:", DATASET_DIR)
print("Image base directory:", IMAGE_BASE_DIR)
print("Phase 1 mask manifest:", PHASE1_MASK_MANIFEST_PATH)
print("Local model cache:", LOCAL_MODEL_CACHE_DIR)
print("Test CSV:", TEST_CSV_PATH, TEST_CSV_PATH.exists())


Mounted at /content/drive
Restoring RSNA-BraTS 2D zip from Drive cache: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_1_segmentation_monai_brats/dataset_cache/rsnabrats20212d.zip
Extracting RSNA-BraTS 2D dataset locally. This may take several minutes...
Using extracted image directory: /content/phase05d_global23_rsnabrats20212d
Project directory: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging
Dataset directory: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_3a_brats_vqa_dataset/dataset_btumqa_225k
Image base directory: /content/phase05d_global23_rsnabrats20212d
Phase 1 mask manifest: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_1_segmentation_monai_brats/logs/global_mask_manifest.csv
Local model cache: /content/phase05d_global23_model_cache
Test CSV

## Select This Notebook's BiomedCLIP Variant


In [ ]:
BASELINE_VARIANT = {
    "model_key": "biomedclip_global23_dual_view_naturalized",
    "model_label": "BiomedCLIP Dual-View Naturalized Global-23 Ensemble",
    "baseline_type": "frozen_image_text_matching_global23_dual_view_naturalized",
    "visual_support": "raw_montage_plus_monai_mask_overlay",
    "image_input_type": "dual_four_modality_montage_raw_and_monai_mask_overlay",
    "output_dir": PHASE5D_BASE_DIR / "phase05d_a3_biomedclip_global23_dual_view_naturalized",
    "use_monai_overlay": True,
    "use_dual_view": True,
    "use_naturalized_candidates": True,
    "candidate_protocol": "global_23_clean_naturalized_text",
    "candidate_text_protocol": "naturalized_answer_phrases_with_overlay_color_legend",
    "default_run_mode": "full",
}
print("Configured variant:", BASELINE_VARIANT["model_key"])


Configured variant: biomedclip_global23_dual_view_naturalized


## Configure Clean Global-23 BiomedCLIP Run


In [ ]:
import json
import math
import random
import re
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

RUN_MODE = BASELINE_VARIANT.get("default_run_mode", "full")  # "diagnostic", "speed_test", "screening", or "full".
SPEED_TEST_SIZE = 300
SCREENING_SIZE = 2300
BATCH_SIZE = 16
RESUME_FROM_EXISTING = True
CHECKPOINT_EVERY_ROWS = 500
MONTAGE_FORCE_REBUILD = False
ALLOW_DELETE_STALE_PHASE05D_ARTIFACTS = False
MASK_COPY_RETRIES = 3
MASK_PREFLIGHT_MAX_ROWS = 300

PHASE_NAME = "Phase 05D_A - BiomedCLIP Global-23 Optimized Baseline"
MODEL_ID = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
MODEL_LABEL = "BiomedCLIP PubMedBERT ViT-B/16"
BIOMEDCLIP_CONTEXT_LENGTH = 256

MODALITIES = ["flair", "t1", "t1ce", "t2"]
MODALITY_LABELS = {"flair": "FLAIR", "t1": "T1", "t1ce": "T1ce", "t2": "T2"}
MONTAGE_TILE_SIZE = 224
MONTAGE_LABEL_HEIGHT = 24

REPORTING_METADATA_COLS = [
    "question_family",
    "question_style",
    "difficulty_level",
    "ambiguity_flag",
    "signal_gap_bucket",
    "region_target_primary",
    "region_target_secondary",
]

FORBIDDEN_PREDICTION_METADATA = [
    "question_family",
    "question_style",
    "difficulty_level",
    "ambiguity_flag",
    "signal_gap_bucket",
    "region_target_primary",
    "region_target_secondary",
    "decision_rule_id",
    "label_provenance",
    "candidate_keep_reason",
]

PROMPT_TEMPLATES_RAW = [
    "A four-panel brain MRI montage with FLAIR, T1, T1ce, and T2 views. Question: {question} Answer: {candidate}.",
    "Based on this brain tumor MRI image, answer the question: {question}. The answer is {candidate}.",
    "Brain MRI visual question answering. Question: {question}. Candidate answer: {candidate}.",
]

PROMPT_TEMPLATES_OVERLAY = [
    "A four-panel brain MRI montage with FLAIR, T1, T1ce, and T2 views. Colored MONAI segmentation overlays mark tumor subregions. Question: {question} Answer: {candidate}.",
    "Based on this brain tumor MRI image with MONAI tumor-mask overlay, answer the question: {question}. The answer is {candidate}.",
    "Brain MRI visual question answering with colored tumor segmentation support. Question: {question}. Candidate answer: {candidate}.",
]

PROMPT_TEMPLATES_OVERLAY_NATURALIZED = [
    "A four-panel brain MRI montage with FLAIR, T1, T1ce, and T2 views. Colored MONAI segmentation overlays mark tumor subregions. Overlay legend: yellow marks edema, blue marks necrotic or non-enhancing tumor core, and red marks enhancing tumor. Question: {question} Candidate answer: {candidate}.",
    "Based on this brain tumor MRI image with MONAI tumor-mask overlay, use the color legend: yellow edema, blue necrotic or non-enhancing core, red enhancing tumor. Question: {question}. The answer is {candidate}.",
    "Brain MRI visual question answering with colored tumor segmentation support. Yellow indicates edema, blue indicates necrotic or non-enhancing tumor core, and red indicates enhancing tumor. Question: {question}. Candidate answer: {candidate}.",
]

PROMPT_TEMPLATES_RAW_NATURALIZED = [
    "A four-panel brain MRI montage with FLAIR, T1, T1ce, and T2 views. Question: {question} Candidate answer: {candidate}.",
    "Based on this brain tumor MRI image, answer the question using the candidate answer phrase. Question: {question}. Candidate answer: {candidate}.",
    "Brain MRI visual question answering. Question: {question}. Candidate answer: {candidate}.",
]

NATURALIZED_ANSWER_TEXT = {
    "ambiguous": "ambiguous regional evidence",
    "close_gap": "close uncertainty or reliability gap",
    "confident_present": "region confidently present",
    "consistent": "consistent regional evidence",
    "context": "surrounding brain context",
    "distinct": "distinct regional evidence",
    "edema": "edema region",
    "enhancing": "enhancing tumor region",
    "global": "global image context",
    "large_confident": "large confidently identified region",
    "large_uncertain": "large uncertain region",
    "moderate_confident": "moderate confidently identified region",
    "moderate_gap": "moderate uncertainty or reliability gap",
    "moderate_uncertain": "moderate uncertain region",
    "ncr_net": "necrotic or non-enhancing tumor core",
    "none": "no relevant tumor region",
    "not_present": "region not present",
    "shifted": "shifted regional evidence",
    "small_confident": "small confidently identified region",
    "small_uncertain": "small uncertain region",
    "tumor": "whole tumor region",
    "uncertain_present": "region uncertainly present",
    "wide_gap": "wide uncertainty or reliability gap",
}


def now_string():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def write_json(path: Path, payload: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def normalize_answer_text(text):
    if text is None or (isinstance(text, float) and math.isnan(text)):
        return ""
    value = str(text).strip().lower()
    value = value.replace("-", "_").replace(" ", "_")
    value = re.sub(r"[^a-z0-9_]+", "", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value


def normalize_dataset_columns(df: pd.DataFrame):
    out = df.copy()
    for col in [
        "qa_id",
        "unique_id",
        "patient_id",
        "slice_id",
        "question",
        "answer",
        *REPORTING_METADATA_COLS,
    ]:
        if col in out.columns:
            out[col] = out[col].fillna("").astype(str)
    out["gold_answer"] = out["answer"].map(normalize_answer_text)
    out["slice_id"] = out["slice_id"].str.zfill(3)
    out["patient_id"] = out["patient_id"].str.zfill(5)
    out["unique_id"] = out["patient_id"] + "_" + out["slice_id"]
    return out


def build_global23_answer_candidates(*frames):
    candidates = sorted(
        {
            normalize_answer_text(answer)
            for frame in frames
            for answer in frame["answer"].dropna().astype(str).tolist()
        }
    )
    if len(candidates) != 23:
        raise ValueError(f"Expected 23 BTUMQA answer classes, found {len(candidates)}: {candidates}")
    return candidates


def choose_eval_rows(test_df: pd.DataFrame):
    if RUN_MODE == "full":
        return test_df.copy().reset_index(drop=True)
    if RUN_MODE == "diagnostic":
        return test_df.sample(n=min(3, len(test_df)), random_state=RANDOM_STATE).reset_index(drop=True)
    if RUN_MODE not in {"speed_test", "screening"}:
        raise ValueError("RUN_MODE must be 'diagnostic', 'speed_test', 'screening', or 'full'.")

    target_size = SPEED_TEST_SIZE if RUN_MODE == "speed_test" else SCREENING_SIZE
    strat_cols = ["question_family", "question_style", "difficulty_level", "signal_gap_bucket", "gold_answer"]
    work = test_df.copy()
    existing_cols = [col for col in strat_cols if col in work.columns]
    if existing_cols:
        work["stratify_key"] = work[existing_cols].fillna("").astype(str).agg("|".join, axis=1)
        groups = list(work.groupby("stratify_key", sort=False))
        rng = np.random.default_rng(RANDOM_STATE)
        selected_indices = []
        min_per_group = max(1, target_size // max(1, len(groups)))
        for _, group in groups:
            take = min(len(group), min_per_group)
            if take > 0:
                selected_indices.extend(rng.choice(group.index.to_numpy(), size=take, replace=False).tolist())
        remaining = target_size - len(selected_indices)
        if remaining > 0:
            pool = work.loc[~work.index.isin(selected_indices)]
            take = min(remaining, len(pool))
            if take > 0:
                selected_indices.extend(rng.choice(pool.index.to_numpy(), size=take, replace=False).tolist())
        selected = work.loc[selected_indices].sample(frac=1.0, random_state=RANDOM_STATE)
        return selected.drop(columns=["stratify_key"]).reset_index(drop=True)

    return work.sample(n=min(target_size, len(work)), random_state=RANDOM_STATE).reset_index(drop=True)


def image_path_for(row, modality: str):
    patient_id = str(row["patient_id"]).zfill(5)
    slice_id = str(row["slice_id"]).zfill(3)
    return IMAGE_BASE_DIR / modality / f"{modality}_BraTS2021_{patient_id}_{slice_id}.png"


def montage_path_for(row, use_monai_overlay: bool):
    patient_id = str(row["patient_id"]).zfill(5)
    slice_id = str(row["slice_id"]).zfill(3)
    prefix = "monai_overlay_montage" if use_monai_overlay else "raw_montage"
    return LOCAL_MONTAGE_DIR / prefix / f"{prefix}_BraTS2021_{patient_id}_{slice_id}.png"


def load_grayscale_tile(path: Path):
    image = Image.open(path).convert("L")
    image = image.resize((MONTAGE_TILE_SIZE, MONTAGE_TILE_SIZE), resample=Image.BICUBIC)
    return image.convert("RGB")


def load_mask_manifest():
    if not PHASE1_MASK_MANIFEST_PATH.exists():
        raise FileNotFoundError(f"Phase 1 mask manifest not found: {PHASE1_MASK_MANIFEST_PATH}")
    manifest = pd.read_csv(PHASE1_MASK_MANIFEST_PATH)
    required = {"unique_id", "mask_path"}
    missing_cols = sorted(required - set(manifest.columns))
    if missing_cols:
        raise ValueError(f"Phase 1 mask manifest missing columns: {missing_cols}")
    manifest["unique_id"] = manifest["unique_id"].astype(str)
    manifest["mask_path"] = manifest["mask_path"].astype(str)
    return manifest.drop_duplicates("unique_id").set_index("unique_id")["mask_path"].to_dict()


def mask_path_for(row, mask_lookup=None):
    unique_id = str(row["unique_id"])
    if mask_lookup is not None and unique_id in mask_lookup:
        manifest_path = Path(mask_lookup[unique_id])
        try:
            if manifest_path.exists():
                return manifest_path
        except OSError:
            pass
        manifest_name_path = PHASE1_MASK_DIR / manifest_path.name
        try:
            if manifest_name_path.exists():
                return manifest_name_path
        except OSError:
            pass
    patient_id = str(row["patient_id"]).zfill(5)
    slice_id = str(row["slice_id"]).zfill(3)
    return PHASE1_MASK_DIR / f"{patient_id}_{slice_id}.png"


def restore_mask_to_local_cache(mask_path: Path):
    LOCAL_MASK_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    local_path = LOCAL_MASK_CACHE_DIR / mask_path.name
    if local_path.exists() and local_path.stat().st_size > 0:
        return local_path
    last_error = None
    for attempt in range(1, MASK_COPY_RETRIES + 1):
        try:
            if not mask_path.exists():
                raise FileNotFoundError(f"Mask path not found: {mask_path}")
            shutil.copy2(mask_path, local_path)
            if local_path.exists() and local_path.stat().st_size > 0:
                return local_path
        except OSError as exc:
            last_error = exc
            print(f"Mask copy attempt {attempt}/{MASK_COPY_RETRIES} failed for {mask_path}: {exc}")
            time.sleep(2 * attempt)
            try:
                ensure_drive_connection(PROJECT_DRIVE_DIR)
            except Exception:
                pass
    if last_error is not None:
        raise last_error
    raise FileNotFoundError(f"Could not restore mask to local cache: {mask_path}")


def load_mask_tile(mask_path: Path):
    local_mask_path = restore_mask_to_local_cache(mask_path)
    with Image.open(local_mask_path) as mask_img:
        mask = mask_img.convert("L")
        mask = mask.resize((MONTAGE_TILE_SIZE, MONTAGE_TILE_SIZE), resample=Image.NEAREST)
        return np.asarray(mask, dtype=np.uint8)


def apply_monai_mask_overlay(tile: Image.Image, mask_np: np.ndarray, alpha=0.42):
    base = np.asarray(tile.convert("RGB"), dtype=np.float32)
    overlay = base.copy()
    color_map = {
        1: np.asarray([255, 215, 0], dtype=np.float32),
        2: np.asarray([0, 153, 255], dtype=np.float32),
        4: np.asarray([255, 64, 64], dtype=np.float32),
    }
    for label_value, color in color_map.items():
        label_mask = mask_np == label_value
        overlay[label_mask] = (1.0 - alpha) * base[label_mask] + alpha * color
    out = Image.fromarray(np.clip(overlay, 0, 255).astype(np.uint8))
    tumor_mask = mask_np > 0
    if tumor_mask.any():
        draw = ImageDraw.Draw(out)
        padded = np.pad(tumor_mask, pad_width=1, mode="constant", constant_values=False)
        interior = (
            padded[1:-1, 1:-1]
            & padded[:-2, 1:-1]
            & padded[2:, 1:-1]
            & padded[1:-1, :-2]
            & padded[1:-1, 2:]
        )
        ys, xs = np.where(tumor_mask & ~interior)
        for x, y in zip(xs.tolist(), ys.tolist()):
            draw.point((int(x), int(y)), fill=(255, 255, 255))
    return out


def build_montage(row, use_monai_overlay=False, mask_lookup=None, force_rebuild=False):
    out_path = montage_path_for(row, use_monai_overlay=use_monai_overlay)
    if out_path.exists() and not force_rebuild:
        return out_path

    missing = [str(image_path_for(row, modality)) for modality in MODALITIES if not image_path_for(row, modality).exists()]
    if missing:
        raise FileNotFoundError("Missing modality image(s):\n" + "\n".join(missing))

    mask_np = None
    if use_monai_overlay:
        mask_path = mask_path_for(row, mask_lookup=mask_lookup)
        if not mask_path.exists():
            raise FileNotFoundError(f"Missing Phase 1 MONAI mask for {row['unique_id']}: {mask_path}")
        mask_np = load_mask_tile(mask_path)

    width = MONTAGE_TILE_SIZE * 2
    height = (MONTAGE_TILE_SIZE + MONTAGE_LABEL_HEIGHT) * 2
    canvas = Image.new("RGB", (width, height), color=(0, 0, 0))
    draw = ImageDraw.Draw(canvas)

    for idx, modality in enumerate(MODALITIES):
        tile = load_grayscale_tile(image_path_for(row, modality))
        if use_monai_overlay:
            tile = apply_monai_mask_overlay(tile, mask_np)
        x = (idx % 2) * MONTAGE_TILE_SIZE
        y = (idx // 2) * (MONTAGE_TILE_SIZE + MONTAGE_LABEL_HEIGHT)
        canvas.paste(tile, (x, y + MONTAGE_LABEL_HEIGHT))
        draw.rectangle([x, y, x + MONTAGE_TILE_SIZE, y + MONTAGE_LABEL_HEIGHT], fill=(20, 20, 20))
        suffix = " + mask" if use_monai_overlay else ""
        draw.text((x + 8, y + 5), f"{MODALITY_LABELS[modality]}{suffix}", fill=(255, 255, 255))

    out_path.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(out_path)
    return out_path


def variant_dirs(variant):
    base = variant["output_dir"]
    dirs = {
        "base": base,
        "predictions": base / "predictions",
        "checkpoints": base / "predictions" / f"{RUN_MODE}_checkpoints",
        "tables": base / "tables",
        "figures": base / "figures",
        "reports": base / "reports",
        "diagnostics": base / "diagnostics",
        "done": base / "done",
    }
    for path in dirs.values():
        path.mkdir(parents=True, exist_ok=True)
    return dirs


def prediction_output_path(dirs):
    filename = "test_predictions.csv" if RUN_MODE == "full" else f"{RUN_MODE}_predictions.csv"
    return dirs["predictions"] / filename


## Load BTUMQA Rows And Fixed 23-Answer Candidate Set


In [ ]:
test_df = normalize_dataset_columns(pd.read_csv(TEST_CSV_PATH))
train_df = normalize_dataset_columns(pd.read_csv(TRAIN_CSV_PATH)) if TRAIN_CSV_PATH.exists() else test_df.iloc[:0].copy()
val_df = normalize_dataset_columns(pd.read_csv(VAL_CSV_PATH)) if VAL_CSV_PATH.exists() else test_df.iloc[:0].copy()

GLOBAL23_CANDIDATES = build_global23_answer_candidates(train_df, val_df, test_df)
eval_df = choose_eval_rows(test_df)
mask_lookup = load_mask_manifest() if BASELINE_VARIANT["use_monai_overlay"] else {}

input_policy = {
    "phase": PHASE_NAME,
    "created_at": now_string(),
    "candidate_protocol": BASELINE_VARIANT.get("candidate_protocol", "global_23_clean"),
    "candidate_count": len(GLOBAL23_CANDIDATES),
    "candidate_source": "dataset_level_union_of_train_val_test_answers_no_row_fallback",
    "candidate_text_protocol": BASELINE_VARIANT.get("candidate_text_protocol", "normalized_label_text"),
    "allowed_prediction_inputs": [
        "raw_question_text",
        "four_modality_mri_montage_pixels",
        "fixed_global_23_answer_candidate_text",
        "phase_1_monai_mask_pixels_rendered_into_image_for_overlay_variant",
    ],
    "forbidden_prediction_inputs": FORBIDDEN_PREDICTION_METADATA + ["row_level_answer", "row_level_gold_answer"],
    "metadata_policy": "metadata_columns_may_be_saved_for_reporting_only_after_prediction",
    "prompt_protocol": "single_prompt_production_debug_prompt_ensemble_diagnostic",
    "run_mode": RUN_MODE,
    "batch_size": BATCH_SIZE,
    "checkpoint_every_rows": CHECKPOINT_EVERY_ROWS,
}

missing_image_rows = []
for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Checking image paths"):
    missing = [modality for modality in MODALITIES if not image_path_for(row, modality).exists()]
    if missing:
        missing_image_rows.append({"qa_id": row["qa_id"], "unique_id": row["unique_id"], "missing": "|".join(missing)})
if missing_image_rows:
    raise FileNotFoundError(f"Missing modality images for {len(missing_image_rows)} evaluation rows.")

missing_mask_rows = []
if BASELINE_VARIANT["use_monai_overlay"]:
    if RUN_MODE in {"diagnostic", "speed_test"}:
        mask_check_df = eval_df
    else:
        mask_check_df = eval_df.head(min(MASK_PREFLIGHT_MAX_ROWS, len(eval_df)))
        print(
            f"Checking MONAI mask paths for the first {len(mask_check_df)} rows only. "
            "Full overlay masks are restored lazily with retries during inference."
        )
    for _, row in tqdm(mask_check_df.iterrows(), total=len(mask_check_df), desc="Checking MONAI mask paths"):
        mask_path = mask_path_for(row, mask_lookup=mask_lookup)
        try:
            mask_exists = mask_path.exists()
        except OSError as exc:
            print(f"Skipping transient mask exists() error for {mask_path}: {exc}")
            mask_exists = True
        if not mask_exists:
            missing_mask_rows.append({"qa_id": row["qa_id"], "unique_id": row["unique_id"], "missing_mask": str(mask_path)})
    if missing_mask_rows:
        preview = missing_mask_rows[:5]
        raise FileNotFoundError(
            f"Missing MONAI masks for {len(missing_mask_rows)} evaluation rows. Examples: {preview}"
        )

print("Run mode:", RUN_MODE)
print("Variant:", BASELINE_VARIANT["model_key"])
print("Evaluation rows:", len(eval_df))
print("Global-23 candidates:", GLOBAL23_CANDIDATES)
print("Batch size:", BATCH_SIZE)
print("Checkpoint every rows:", CHECKPOINT_EVERY_ROWS)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
display(eval_df[["qa_id", "unique_id", "question", "gold_answer"] + [c for c in REPORTING_METADATA_COLS if c in eval_df.columns]].head())


Checking image paths:   0%|          | 0/33750 [00:00<?, ?it/s]

Checking MONAI mask paths for the first 300 rows only. Full overlay masks are restored lazily with retries during inference.


Checking MONAI mask paths:   0%|          | 0/300 [00:00<?, ?it/s]

Run mode: full
Variant: biomedclip_global23_dual_view_naturalized
Evaluation rows: 33750
Global-23 candidates: ['ambiguous', 'close_gap', 'confident_present', 'consistent', 'context', 'distinct', 'edema', 'enhancing', 'global', 'large_confident', 'large_uncertain', 'moderate_confident', 'moderate_gap', 'moderate_uncertain', 'ncr_net', 'none', 'not_present', 'shifted', 'small_confident', 'small_uncertain', 'tumor', 'uncertain_present', 'wide_gap']
Batch size: 16
Checkpoint every rows: 500
Device: cuda


,qa_id,unique_id,question,gold_answer,question_family,question_style,difficulty_level,ambiguity_flag,signal_gap_bucket,region_target_primary,region_target_secondary
0,BTUMQA225K_07a3eda3b59c63,00002_033,"Is edema confidently present, uncertainly pres...",not_present,confidence_qualified_presence,confidence_qualified,easy,no,wide_gap,edema,
1,BTUMQA225K_668ac232b10861,00002_033,Which region is safest to trust for reasoning ...,global,safe_region_for_reasoning,ranking,easy,no,wide_gap,global,
2,BTUMQA225K_f1d4b092590518,00002_033,Which confidence-qualified extent label best f...,none,confidence_qualified_extent,confidence_qualified,easy,no,wide_gap,tumor,
3,BTUMQA225K_c2e575cfe72f68,00002_034,Which region should receive the highest trust ...,global,safe_region_for_reasoning,ranking,easy,no,wide_gap,global,
4,BTUMQA225K_142f3c4ec5e345,00002_035,"Is edema confidently present, uncertainly pres...",not_present,confidence_qualified_presence,confidence_qualified,easy,no,wide_gap,edema,


## Optional Cleanup For Old Oversized Artifacts


In [ ]:
def list_stale_phase05d_artifacts(variant):
    dirs = variant_dirs(variant)
    candidates = []
    patterns = [
        "test_predictions.csv.tmp",
        "*_predictions.csv.tmp",
        "*.tmp",
    ]
    for pattern in patterns:
        candidates.extend(dirs["predictions"].glob(pattern))
    oversized_predictions = [
        path
        for path in dirs["predictions"].glob("*predictions*.csv")
        if path.is_file() and path.stat().st_size > 100 * 1024 * 1024
    ]
    candidates.extend(oversized_predictions)

    rows = []
    seen = set()
    for path in sorted(candidates):
        if path in seen or not path.exists() or not path.is_file():
            continue
        seen.add(path)
        rows.append(
            {
                "path": str(path),
                "size_mb": round(path.stat().st_size / (1024 * 1024), 2),
                "last_modified": datetime.fromtimestamp(path.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
            }
        )
    return pd.DataFrame(rows)


stale_artifacts_df = list_stale_phase05d_artifacts(BASELINE_VARIANT)
if stale_artifacts_df.empty:
    print("No stale oversized Phase 05D temporary/prediction artifacts found for this variant.")
else:
    display(stale_artifacts_df)
    if ALLOW_DELETE_STALE_PHASE05D_ARTIFACTS:
        for path_text in stale_artifacts_df["path"].tolist():
            path = Path(path_text)
            if path.exists() and path.is_file() and str(path).startswith(str(BASELINE_VARIANT["output_dir"])):
                path.unlink()
                print("Deleted:", path)
    else:
        print("Deletion is disabled. Set ALLOW_DELETE_STALE_PHASE05D_ARTIFACTS = True in the config cell to delete listed files.")


No stale oversized Phase 05D temporary/prediction artifacts found for this variant.


## Run Diagnostic, Speed Test, Screening, Or Full Inference


In [ ]:
import time

import open_clip


def load_biomedclip_model():
    model, preprocess = open_clip.create_model_from_pretrained(MODEL_ID)
    tokenizer = open_clip.get_tokenizer(MODEL_ID)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()
    return model, preprocess, tokenizer, device


def prompt_templates_for_variant(variant):
    if variant.get("use_dual_view", False):
        templates = PROMPT_TEMPLATES_OVERLAY_NATURALIZED
    elif variant.get("use_naturalized_candidates", False):
        templates = PROMPT_TEMPLATES_OVERLAY_NATURALIZED
    else:
        templates = PROMPT_TEMPLATES_OVERLAY if variant["use_monai_overlay"] else PROMPT_TEMPLATES_RAW
    if RUN_MODE == "diagnostic":
        return templates
    return templates[:1]


def prompt_protocol_for_run():
    if BASELINE_VARIANT.get("use_dual_view", False):
        return "dual_view_naturalized_prompt_ensemble_debug" if RUN_MODE == "diagnostic" else "dual_view_naturalized_single_prompt"
    return "prompt_ensemble_mean_logits_debug" if RUN_MODE == "diagnostic" else "single_prompt_global23"


def prompt_templates_for_view(variant, view_name):
    if variant.get("use_dual_view", False):
        templates = PROMPT_TEMPLATES_RAW_NATURALIZED if view_name == "raw" else PROMPT_TEMPLATES_OVERLAY_NATURALIZED
    else:
        templates = prompt_templates_for_variant(variant)
    if RUN_MODE == "diagnostic":
        return templates
    return templates[:1]


def candidate_text_for_variant(candidate, variant):
    if variant.get("use_naturalized_candidates", False):
        return NATURALIZED_ANSWER_TEXT.get(candidate, candidate.replace("_", " "))
    return candidate


def build_candidate_texts(question, candidates, prompt_templates, variant):
    texts = []
    for candidate in candidates:
        candidate_text = candidate_text_for_variant(candidate, variant)
        for template in prompt_templates:
            texts.append(template.format(question=question, candidate=candidate_text))
    return texts


def score_candidate_logits_for_block(block_logits, candidate_count, template_count):
    block = np.asarray(block_logits, dtype=np.float64).reshape(candidate_count, template_count)
    return block.mean(axis=1)


def softmax_np(values):
    shifted = values - np.max(values)
    probs = np.exp(shifted)
    return probs / probs.sum()


def run_biomedclip_batch(model, preprocess, tokenizer, device, records, candidates, prompt_templates, variant):
    if len(candidates) != 23:
        raise ValueError(f"Global-23 protocol violation: expected 23 candidates, got {len(candidates)}")
    if not records:
        return []

    images = []
    texts = []
    image_text_blocks = []
    for record in records:
        for view in record["views"]:
            view_templates = prompt_templates_for_view(variant, view["view_name"])
            block_start = len(texts)
            block_texts = build_candidate_texts(record["question"], candidates, view_templates, variant)
            texts.extend(block_texts)
            images.append(preprocess(Image.open(view["montage_path"]).convert("RGB")))
            image_text_blocks.append(
                {
                    "qa_id": record["row"]["qa_id"],
                    "view_name": view["view_name"],
                    "text_start": block_start,
                    "text_end": block_start + len(block_texts),
                    "template_count": len(view_templates),
                }
            )

    image_tensor = torch.stack(images, dim=0).to(device)
    tokenized = tokenizer(texts, context_length=BIOMEDCLIP_CONTEXT_LENGTH).to(device)

    with torch.no_grad():
        image_features, text_features, logit_scale = model(image_tensor, tokenized)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        all_logits = (logit_scale * image_features @ text_features.t()).detach().cpu().numpy()

    candidate_count = len(candidates)
    image_cursor = 0
    outputs = []
    for row_index, record in enumerate(records):
        view_logits = []
        view_names = []
        for view in record["views"]:
            block = image_text_blocks[image_cursor]
            candidate_logits_for_view = score_candidate_logits_for_block(
                all_logits[image_cursor, block["text_start"] : block["text_end"]],
                candidate_count,
                block["template_count"],
            )
            view_logits.append(candidate_logits_for_view)
            view_names.append(view["view_name"])
            image_cursor += 1
        candidate_logits = np.stack(view_logits, axis=0).mean(axis=0)
        candidate_probs = softmax_np(candidate_logits)
        best_idx = int(candidate_probs.argmax())
        output = {
            "predicted_answer": candidates[best_idx],
            "confidence": float(candidate_probs[best_idx]),
            "candidate_probabilities": candidate_probs,
            "candidate_logits": candidate_logits,
            "view_names": view_names,
            "view_count": len(view_names),
        }
        outputs.append(output)
    return outputs


def prediction_payload(row, variant, predicted_answer, confidence, parse_status, debug_payload=None):
    normalized_pred = normalize_answer_text(predicted_answer)
    gold = normalize_answer_text(row["gold_answer"])
    payload = {
        "qa_id": row["qa_id"],
        "unique_id": row["unique_id"],
        "patient_id": row["patient_id"],
        "slice_id": row["slice_id"],
        "gold_answer": gold,
        "predicted_answer": normalized_pred,
        "is_correct": int(normalized_pred == gold),
        "confidence": confidence,
        "model_key": variant["model_key"],
        "model_name": variant["model_label"],
        "baseline_type": variant["baseline_type"],
        "image_input_type": variant["image_input_type"],
        "visual_support": variant["visual_support"],
        "candidate_protocol": variant.get("candidate_protocol", "global_23_clean"),
        "candidate_count": len(GLOBAL23_CANDIDATES),
        "candidate_text_protocol": variant.get("candidate_text_protocol", "normalized_label_text"),
        "prompt_protocol": prompt_protocol_for_run(),
        "prompt_template_count": len(prompt_templates_for_variant(variant)),
        "metadata_policy": "no_question_family_or_bias_metadata_for_prediction",
        "answer_parse_status": parse_status,
    }
    for col in REPORTING_METADATA_COLS:
        payload[col] = row.get(col, "")
    if RUN_MODE == "diagnostic":
        payload["question"] = row.get("question", "")
        payload["raw_output"] = json.dumps(debug_payload or {})
    return payload


def load_existing_prediction_rows(output_path: Path, checkpoint_dir: Path):
    frames = []
    if output_path.exists():
        frames.append(pd.read_csv(output_path))
    if checkpoint_dir.exists():
        for part_path in sorted(checkpoint_dir.glob("checkpoint_*.csv")):
            try:
                frames.append(pd.read_csv(part_path))
            except Exception as exc:
                print("Skipping unreadable checkpoint:", part_path, type(exc).__name__, exc)
    if not frames:
        return []
    out_df = pd.concat(frames, ignore_index=True)
    if "qa_id" in out_df.columns:
        out_df["qa_id"] = out_df["qa_id"].astype(str)
        out_df = out_df.drop_duplicates("qa_id", keep="last")
    return out_df.to_dict("records")


def next_checkpoint_index(checkpoint_dir: Path):
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    existing = sorted(checkpoint_dir.glob("checkpoint_*.csv"))
    if not existing:
        return 1
    last = existing[-1].stem.split("_")[-1]
    return int(last) + 1 if last.isdigit() else len(existing) + 1


def save_checkpoint_rows(rows, checkpoint_dir: Path, checkpoint_index: int):
    if not rows:
        return None
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    path = checkpoint_dir / f"checkpoint_{checkpoint_index:05d}.csv"
    pd.DataFrame(rows).to_csv(path, index=False)
    print("Saved compact checkpoint:", path, "rows:", len(rows))
    return path


def save_final_predictions(rows, output_path: Path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    out_df = pd.DataFrame(rows)
    if "qa_id" in out_df.columns:
        out_df["qa_id"] = out_df["qa_id"].astype(str)
        out_df = out_df.drop_duplicates("qa_id", keep="last")
    out_df.to_csv(output_path, index=False)
    return out_df


def build_record(row, variant):
    if variant.get("use_dual_view", False):
        views = [
            {
                "view_name": "raw",
                "montage_path": build_montage(
                    row,
                    use_monai_overlay=False,
                    mask_lookup=mask_lookup,
                    force_rebuild=MONTAGE_FORCE_REBUILD,
                ),
            },
            {
                "view_name": "monai_overlay",
                "montage_path": build_montage(
                    row,
                    use_monai_overlay=True,
                    mask_lookup=mask_lookup,
                    force_rebuild=MONTAGE_FORCE_REBUILD,
                ),
            },
        ]
    else:
        views = [
            {
                "view_name": "monai_overlay" if variant["use_monai_overlay"] else "raw",
                "montage_path": build_montage(
                    row,
                    use_monai_overlay=variant["use_monai_overlay"],
                    mask_lookup=mask_lookup,
                    force_rebuild=MONTAGE_FORCE_REBUILD,
                ),
            }
        ]
    return {
        "row": row,
        "question": row["question"],
        "views": views,
    }


def payloads_for_records(records, outputs, variant, prompt_templates):
    rows = []
    for record, result in zip(records, outputs):
        debug_payload = None
        if RUN_MODE == "diagnostic":
            debug_payload = {
                "best_answer": result["predicted_answer"],
                "best_answer_text": candidate_text_for_variant(result["predicted_answer"], variant),
                "best_probability": result["confidence"],
                "candidate_protocol": variant.get("candidate_protocol", "global_23_clean"),
                "candidate_text_protocol": variant.get("candidate_text_protocol", "normalized_label_text"),
                "candidate_count": int(len(GLOBAL23_CANDIDATES)),
                "prompt_template_count": int(len(prompt_templates)),
                "view_count": int(result.get("view_count", 1)),
                "view_names": result.get("view_names", []),
                "candidate_texts": {
                    str(candidate): candidate_text_for_variant(candidate, variant)
                    for candidate in GLOBAL23_CANDIDATES
                },
                "candidate_probabilities": {
                    str(candidate): float(prob)
                    for candidate, prob in zip(GLOBAL23_CANDIDATES, result["candidate_probabilities"].tolist())
                },
                "candidate_mean_logits": {
                    str(candidate): float(logit)
                    for candidate, logit in zip(GLOBAL23_CANDIDATES, result["candidate_logits"].tolist())
                },
            }
        rows.append(
            prediction_payload(
                row=record["row"],
                variant=variant,
                predicted_answer=result["predicted_answer"],
                confidence=result["confidence"],
                parse_status="candidate_similarity_global23_batched",
                debug_payload=debug_payload,
            )
        )
    return rows


def error_payload(row, variant, exc):
    return prediction_payload(
        row=row,
        variant=variant,
        predicted_answer="",
        confidence=np.nan,
        parse_status=f"error:{type(exc).__name__}:{exc}",
        debug_payload={"error": f"{type(exc).__name__}: {exc}"},
    )


def infer_records_with_fallback(model, preprocess, tokenizer, device, records, variant, prompt_templates):
    if not records:
        return []
    try:
        outputs = run_biomedclip_batch(
            model=model,
            preprocess=preprocess,
            tokenizer=tokenizer,
            device=device,
            records=records,
            candidates=GLOBAL23_CANDIDATES,
            prompt_templates=prompt_templates,
            variant=variant,
        )
        return payloads_for_records(records, outputs, variant, prompt_templates)
    except RuntimeError as exc:
        if "out of memory" in str(exc).lower() and torch.cuda.is_available():
            torch.cuda.empty_cache()
        print("Batch inference failed; retrying rows one-by-one:", type(exc).__name__, exc)
        payloads = []
        for record in records:
            try:
                outputs = run_biomedclip_batch(
                    model=model,
                    preprocess=preprocess,
                    tokenizer=tokenizer,
                    device=device,
                    records=[record],
                    candidates=GLOBAL23_CANDIDATES,
                    prompt_templates=prompt_templates,
                    variant=variant,
                )
                payloads.extend(payloads_for_records([record], outputs, variant, prompt_templates))
            except Exception as row_exc:
                payloads.append(error_payload(record["row"], variant, row_exc))
        return payloads


def run_variant(variant, model, preprocess, tokenizer, device):
    dirs = variant_dirs(variant)
    output_path = prediction_output_path(dirs)
    checkpoint_dir = dirs["checkpoints"]
    done_path = dirs["done"] / f"{variant['model_key']}_{RUN_MODE}_complete.json"
    input_policy_path = dirs["reports"] / f"{variant['model_key']}_{RUN_MODE}_input_policy.json"
    write_json(input_policy_path, input_policy | {"variant": variant["model_key"]})
    eval_manifest_path = dirs["tables"] / f"{variant['model_key']}_{RUN_MODE}_eval_manifest.csv"
    eval_df.to_csv(eval_manifest_path, index=False)

    existing_rows = load_existing_prediction_rows(output_path, checkpoint_dir) if RESUME_FROM_EXISTING else []
    done_qa_ids = {str(row["qa_id"]) for row in existing_rows if "qa_id" in row}
    if existing_rows:
        print("Resuming", variant["model_key"], "from compact rows:", len(existing_rows))

    prompt_templates = prompt_templates_for_variant(variant)
    print("Prompt template count for this run:", len(prompt_templates))
    prediction_rows = list(existing_rows)
    checkpoint_buffer = []
    checkpoint_index = next_checkpoint_index(checkpoint_dir)
    pending_df = eval_df[~eval_df["qa_id"].astype(str).isin(done_qa_ids)].reset_index(drop=True)
    started_at = time.time()

    batch_starts = range(0, len(pending_df), BATCH_SIZE)
    progress = tqdm(total=len(pending_df), desc=variant["model_key"], unit="row")
    processed_since_checkpoint = 0
    for start in batch_starts:
        batch_df = pending_df.iloc[start : start + BATCH_SIZE]
        records = []
        batch_payloads = []
        for _, row in batch_df.iterrows():
            try:
                records.append(build_record(row, variant))
            except Exception as exc:
                batch_payloads.append(error_payload(row, variant, exc))

        batch_payloads.extend(
            infer_records_with_fallback(
                model=model,
                preprocess=preprocess,
                tokenizer=tokenizer,
                device=device,
                records=records,
                variant=variant,
                prompt_templates=prompt_templates,
            )
        )
        prediction_rows.extend(batch_payloads)
        checkpoint_buffer.extend(batch_payloads)
        processed_since_checkpoint += len(batch_payloads)
        progress.update(len(batch_payloads))

        if processed_since_checkpoint >= CHECKPOINT_EVERY_ROWS:
            save_checkpoint_rows(checkpoint_buffer, checkpoint_dir, checkpoint_index)
            checkpoint_index += 1
            checkpoint_buffer = []
            processed_since_checkpoint = 0
    progress.close()

    if checkpoint_buffer:
        save_checkpoint_rows(checkpoint_buffer, checkpoint_dir, checkpoint_index)

    pred_df = save_final_predictions(prediction_rows, output_path)
    valid_df = pred_df[~pred_df["answer_parse_status"].astype(str).str.startswith("error")].copy()
    elapsed_minutes = (time.time() - started_at) / 60.0
    report = {
        "finished_at": now_string(),
        "phase": PHASE_NAME,
        "run_mode": RUN_MODE,
        "model_key": variant["model_key"],
        "model_label": variant["model_label"],
        "prediction_path": str(output_path),
        "checkpoint_dir": str(checkpoint_dir),
        "num_prediction_rows": int(len(pred_df)),
        "num_valid_rows": int(len(valid_df)),
        "elapsed_minutes_this_session": float(elapsed_minutes),
        "rows_per_minute_this_session": float(len(pending_df) / elapsed_minutes) if elapsed_minutes > 0 else None,
        "batch_size": int(BATCH_SIZE),
        "candidate_protocol": variant.get("candidate_protocol", "global_23_clean"),
        "candidate_count": int(len(GLOBAL23_CANDIDATES)),
        "candidate_text_protocol": variant.get("candidate_text_protocol", "normalized_label_text"),
        "prompt_protocol": prompt_protocol_for_run(),
        "prompt_template_count": int(len(prompt_templates)),
        "view_count": 2 if variant.get("use_dual_view", False) else 1,
        "view_names": ["raw", "monai_overlay"] if variant.get("use_dual_view", False) else (["monai_overlay"] if variant.get("use_monai_overlay", False) else ["raw"]),
        "compact_full_mode": bool(RUN_MODE != "diagnostic"),
        "accuracy": float(accuracy_score(valid_df["gold_answer"], valid_df["predicted_answer"])) if len(valid_df) else None,
        "macro_f1": float(f1_score(valid_df["gold_answer"], valid_df["predicted_answer"], average="macro", zero_division=0)) if len(valid_df) else None,
        "weighted_f1": float(f1_score(valid_df["gold_answer"], valid_df["predicted_answer"], average="weighted", zero_division=0)) if len(valid_df) else None,
        "input_policy_path": str(input_policy_path),
    }
    report_path = dirs["reports"] / f"{variant['model_key']}_{RUN_MODE}_inference_report.json"
    write_json(report_path, report)
    write_json(done_path, {"finished_at": now_string(), "report_path": str(report_path), "prediction_path": str(output_path)})
    print(json.dumps(report, indent=2))
    return report


biomedclip_model, biomedclip_preprocess, biomedclip_tokenizer, biomedclip_device = load_biomedclip_model()
report = run_variant(BASELINE_VARIANT, biomedclip_model, biomedclip_preprocess, biomedclip_tokenizer, biomedclip_device)
print("Completed report:")
print(report["model_key"], report["prediction_path"])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

open_clip_pytorch_model.bin:   0%|          | 0.00/784M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Resuming biomedclip_global23_dual_view_naturalized from compact rows: 22016
Prompt template count for this run: 1


biomedclip_global23_dual_view_naturalized:   0%|          | 0/11734 [00:00<?, ?row/s]

Saved compact checkpoint: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_5d_vlm_global23_prompt_ensemble/phase05d_a3_biomedclip_global23_dual_view_naturalized/predictions/full_checkpoints/checkpoint_00044.csv rows: 512
Saved compact checkpoint: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_5d_vlm_global23_prompt_ensemble/phase05d_a3_biomedclip_global23_dual_view_naturalized/predictions/full_checkpoints/checkpoint_00045.csv rows: 512
Saved compact checkpoint: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_5d_vlm_global23_prompt_ensemble/phase05d_a3_biomedclip_global23_dual_view_naturalized/predictions/full_checkpoints/checkpoint_00046.csv rows: 512
Saved compact checkpoint: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_5d_vlm_global23_prompt_ens